# Chest X-Ray Pneumonia Detection — VGG19

**Dataset:** [Kaggle – Chest X-Ray Images (Pneumonia)](https://www.kaggle.com/datasets/paultimothymooney/chest-xray-pneumonia)  
**Architecture:** VGG19 (ImageNet pre-trained, transfer learning)  
**Task:** Binary classification — NORMAL vs PNEUMONIA

---
### Table of contents
1. Environment setup  
2. Dataset overview & EDA  
3. Data augmentation  
4. Model architecture  
5. Training  
6. Evaluation (Accuracy, Precision, Recall, F1, Confusion Matrix, ROC)  
7. Save & export model

## 1. Environment Setup

In [ ]:
import sys, os, json, random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image

import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, callbacks
from tensorflow.keras.applications import VGG19
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, precision_score, recall_score, f1_score,
    roc_curve, auc,
)

print(f'TensorFlow {tf.__version__}')
print(f'GPUs: {tf.config.list_physical_devices("GPU")}')

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

DATA_DIR   = Path('../data/chest_xray')
OUTPUT_DIR = Path('../ml/outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

IMG_SIZE   = (224, 224)
BATCH_SIZE = 32
EPOCHS     = 20

## 2. Dataset Overview & EDA

In [ ]:
splits  = ['train', 'val', 'test']
classes = ['NORMAL', 'PNEUMONIA']

counts = {}
for split in splits:
    counts[split] = {}
    for cls in classes:
        p = DATA_DIR / split / cls
        n = len(list(p.glob('*.jpeg')) + list(p.glob('*.jpg')))
        counts[split][cls] = n
        print(f'  {split:6s} / {cls:10s}: {n:5d}')

In [ ]:
fig, ax = plt.subplots(figsize=(9,5))
x, w = np.arange(len(splits)), 0.35
ax.bar(x - w/2, [counts[s]['NORMAL']    for s in splits], w, label='Normal',    color='#4A90D9')
ax.bar(x + w/2, [counts[s]['PNEUMONIA'] for s in splits], w, label='Pneumonia', color='#E05252')
ax.set_xticks(x); ax.set_xticklabels([s.title() for s in splits])
ax.set_ylabel('Images'); ax.set_title('Class Distribution per Split', fontweight='bold')
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(18,8))
fig.suptitle('Sample Chest X-Ray Images (Train)', fontsize=14, fontweight='bold')
for row, cls in enumerate(classes):
    imgs = list((DATA_DIR/'train'/cls).glob('*.jpeg'))[:5]
    for col, path in enumerate(imgs):
        axes[row][col].imshow(Image.open(path).convert('L'), cmap='gray')
        axes[row][col].set_title(cls, fontsize=9); axes[row][col].axis('off')
plt.tight_layout(); plt.show()

## 3. Data Augmentation & Generators

In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    fill_mode='nearest',
)
val_test_datagen = ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_directory(
    DATA_DIR/'train', target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='binary', shuffle=True)
val_gen = val_test_datagen.flow_from_directory(
    DATA_DIR/'val',   target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='binary', shuffle=False)
test_gen = val_test_datagen.flow_from_directory(
    DATA_DIR/'test',  target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='binary', shuffle=False)

CLASS_NAMES = list(train_gen.class_indices.keys())
print('Classes:', CLASS_NAMES)

# Class weights to handle imbalance
c = np.bincount(train_gen.classes)
class_wt = {i: c.sum()/(len(c)*ci) for i, ci in enumerate(c)}
print('Class weights:', class_wt)

In [ ]:
# Visualise augmented samples
batch_imgs, batch_labels = next(train_gen)
fig, axes = plt.subplots(2, 4, figsize=(14,6))
fig.suptitle('Augmented Training Samples', fontsize=12, fontweight='bold')
for i, ax in enumerate(axes.ravel()):
    ax.imshow(batch_imgs[i]); ax.axis('off')
    ax.set_title(CLASS_NAMES[int(batch_labels[i])], fontsize=9)
plt.tight_layout(); plt.show()

## 4. Model Architecture (VGG19 Transfer Learning)

In [ ]:
base_model = VGG19(weights='imagenet', include_top=False, input_shape=(*IMG_SIZE, 3))
base_model.trainable = False

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.BatchNormalization(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(1, activation='sigmoid'),
], name='vgg19_pneumonia')

model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')],
)
model.summary()

## 5. Training

In [ ]:
ckpt_path = OUTPUT_DIR / 'best_model.h5'

cb_list = [
    callbacks.ModelCheckpoint(str(ckpt_path), monitor='val_accuracy',
                               save_best_only=True, verbose=1),
    callbacks.EarlyStopping(monitor='val_loss', patience=5,
                             restore_best_weights=True, verbose=1),
    callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                 patience=3, min_lr=1e-7, verbose=1),
]

history = model.fit(
    train_gen,
    epochs=EPOCHS,
    validation_data=val_gen,
    class_weight=class_wt,
    callbacks=cb_list,
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13,5))
fig.suptitle('Training Curves', fontsize=13, fontweight='bold')

axes[0].plot(history.history['accuracy'],     label='Train')
axes[0].plot(history.history['val_accuracy'], label='Val')
axes[0].set_title('Accuracy'); axes[0].set_xlabel('Epoch'); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(history.history['loss'],     label='Train')
axes[1].plot(history.history['val_loss'], label='Val')
axes[1].set_title('Loss'); axes[1].set_xlabel('Epoch'); axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout(); plt.show()

## 6. Evaluation

In [ ]:
model.load_weights(str(ckpt_path))

proba = model.predict(test_gen, verbose=1).ravel()
preds = (proba >= 0.5).astype(int)
true  = test_gen.classes

acc  = accuracy_score(true, preds)
prec = precision_score(true, preds)
rec  = recall_score(true, preds)
f1   = f1_score(true, preds)

print(f'Accuracy  : {acc:.4f}')
print(f'Precision : {prec:.4f}')
print(f'Recall    : {rec:.4f}')
print(f'F1-Score  : {f1:.4f}')
print()
print(classification_report(true, preds, target_names=CLASS_NAMES))

In [ ]:
cm = confusion_matrix(true, preds)
fig, ax = plt.subplots(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
ax.set_title('Confusion Matrix – Test Set', fontweight='bold')
ax.set_ylabel('True'); ax.set_xlabel('Predicted')
plt.tight_layout(); plt.show()

In [ ]:
fpr, tpr, _ = roc_curve(true, proba)
roc_auc     = auc(fpr, tpr)
fig, ax = plt.subplots(figsize=(6,5))
ax.plot(fpr, tpr, lw=2, label=f'AUC = {roc_auc:.4f}')
ax.plot([0,1],[0,1],'--', color='gray')
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve – Test Set', fontweight='bold')
ax.legend(loc='lower right'); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 7. Save & Export Model

In [ ]:
final_path = OUTPUT_DIR / 'pneumonia_vgg19_final.h5'
model.save(str(final_path))
print(f'Model saved → {final_path}')

with open(OUTPUT_DIR / 'class_indices.json', 'w') as f:
    json.dump(train_gen.class_indices, f, indent=2)
print('Class indices saved.')

metrics = dict(accuracy=round(acc,4), precision=round(prec,4),
               recall=round(rec,4), f1=round(f1,4))
with open(OUTPUT_DIR / 'metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print('Metrics saved:', metrics)